In [0]:

# ==================================================
# PHASE 2: FAILURE RECOVERY & WATERMARK TEST
# Separate QA Delta table - production data is safe
# ==================================================

import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

TABLE = "github_health.ops.qa_watermark_recovery"

# -----------------------------------------------
# STEP 1: Create QA watermark table
# -----------------------------------------------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    repository STRING,
    entity STRING,
    layer STRING,
    last_successful_watermark TIMESTAMP,
    batch_id STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

# Use a unique test repository for this run.
# Never touch production watermark rows.
repo = "qa/recovery_" + uuid.uuid4().hex[:8]
entity = "commits"
layer = "RAW"

old_time = datetime(2026, 10, 8, 11, 45, 5)
new_time = datetime(2026, 10, 8, 17, 16, 39)

test_batch = "qa_incremental_001"

SCHEMA = StructType([
    StructField("repository", StringType(), False),
    StructField("entity", StringType(), False),
    StructField("layer", StringType(), False),
    StructField(
        "last_successful_watermark",
        TimestampType(),
        True
    ),
    StructField("batch_id", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])


# -----------------------------------------------
# STEP 2: Watermark MERGE function
# -----------------------------------------------

def update_watermark(checkpoint, batch):

    updated_at = datetime.now(
        timezone.utc
    ).replace(tzinfo=None)

    data = [(
        repo,
        entity,
        layer,
        checkpoint,
        batch,
        updated_at
    )]

    source = spark.createDataFrame(
        data,
        schema=SCHEMA
    )

    (
        DeltaTable.forName(spark, TABLE)
        .alias("target")
        .merge(
            source.alias("source"),
            """
            target.repository = source.repository
            AND target.entity = source.entity
            AND target.layer = source.layer
            """
        )
        .whenMatchedUpdate(
            condition="""
                target.last_successful_watermark
                < source.last_successful_watermark
            """,
            set={
                "last_successful_watermark":
                    "source.last_successful_watermark",
                "batch_id": "source.batch_id",
                "updated_at": "source.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


def get_watermark():

    rows = (
        spark.table(TABLE)
        .filter(
            (F.col("repository") == repo) &
            (F.col("entity") == entity) &
            (F.col("layer") == layer)
        )
        .collect()
    )

    assert len(rows) == 1, (
        f"Expected one watermark, found {len(rows)}"
    )

    return rows[0]


# -----------------------------------------------
# STEP 3: Initialize test watermark
# -----------------------------------------------

print("\nINITIALIZING TEST WATERMARK")

update_watermark(
    old_time,
    "qa_full_baseline"
)

initial = get_watermark()

assert initial["last_successful_watermark"] == old_time

print("Initial watermark:", old_time)


# -----------------------------------------------
# TEST 1: Simulate ingestion failure
# -----------------------------------------------

print("\nTEST 1: SIMULATED INGESTION FAILURE")

try:

    print("Starting simulated incremental ingestion...")

    # Simulate a failed GitHub API request
    raise RuntimeError(
        "Simulated GitHub API failure"
    )

    # This must NEVER execute after the failure
    update_watermark(new_time, test_batch)

except RuntimeError as error:
    print("Expected failure:", error)

after_failure = get_watermark()

assert (
    after_failure["last_successful_watermark"]
    == old_time
)

print("PASS: Failed ingestion did not advance watermark")


# -----------------------------------------------
# TEST 2: Successful retry
# -----------------------------------------------

print("\nTEST 2: SUCCESSFUL RETRY")

# Simulate successful completion of ingestion
print("Simulated raw ingestion completed successfully")

update_watermark(
    new_time,
    test_batch
)

after_success = get_watermark()

assert (
    after_success["last_successful_watermark"]
    == new_time
)

assert after_success["batch_id"] == test_batch

print("PASS: Successful retry advanced watermark")
print("New watermark:", new_time)


# -----------------------------------------------
# TEST 3: Idempotent retry
# -----------------------------------------------

print("\nTEST 3: REPEAT SUCCESSFUL BATCH")

before_replay = get_watermark()

update_watermark(
    new_time,
    test_batch
)

after_replay = get_watermark()

assert (
    after_replay["last_successful_watermark"]
    == new_time
)

assert (
    after_replay["updated_at"]
    == before_replay["updated_at"]
)

assert after_replay["batch_id"] == test_batch

print("PASS: Repeated batch left watermark unchanged")


# -----------------------------------------------
# TEST 4: Duplicate watermark check
# -----------------------------------------------

print("\nTEST 4: DUPLICATE CHECK")

count = (
    spark.table(TABLE)
    .filter(
        (F.col("repository") == repo) &
        (F.col("entity") == entity) &
        (F.col("layer") == layer)
    )
    .count()
)

assert count == 1

print("PASS: Only one watermark row exists")


# -----------------------------------------------
# TEST 5: Stale checkpoint protection
# -----------------------------------------------

print("\nTEST 5: OLDER CHECKPOINT REPLAY")

update_watermark(
    old_time,
    "qa_stale_batch"
)

after_stale = get_watermark()

assert (
    after_stale["last_successful_watermark"]
    == new_time
)

assert after_stale["batch_id"] == test_batch

print("PASS: Older checkpoint did not overwrite newer one")


# -----------------------------------------------
# FINAL SUMMARY
# -----------------------------------------------

print("\n" + "=" * 55)
print("ALL 5 FAILURE RECOVERY TESTS PASSED")
print("=" * 55)

print("Initial watermark :", old_time)
print("Final watermark   :", new_time)
print("Final row count   :", count)
print("Final batch ID    :", after_stale["batch_id"])
print("Production tables : NOT MODIFIED")
